# 🚁 ResQ — Aerial Person Detection Fine-Tuning

This notebook fine-tunes **YOLOv8m** on the **VisDrone** aerial dataset for improved person detection from drone footage.

**Runtime:** Make sure you're using a **GPU runtime**:
- Go to `Runtime → Change runtime type → GPU (T4)` 

**Estimated time:** ~1-2 hours on T4 GPU (50 epochs)

## 1. Install Dependencies

In [ ]:
!pip install -q ultralytics
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU! Go to Runtime → Change runtime type → GPU")

## 2. Fine-Tune on VisDrone (Auto-Downloads ~3 GB)

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8m
model = YOLO("yolov8m.pt")

# Fine-tune on VisDrone with aerial-optimized settings
results = model.train(
    data="VisDrone.yaml",       # Auto-downloads VisDrone dataset
    epochs=50,
    batch=16,                   # Higher batch on GPU
    imgsz=640,
    patience=10,               # Early stopping
    name="yolov8m-aerial",
    project="runs/aerial",
    exist_ok=True,
    # Aerial-optimized augmentations
    augment=True,
    mosaic=1.0,
    mixup=0.15,
    scale=0.7,                 # Aggressive scale jitter for altitude simulation
    fliplr=0.5,
    flipud=0.2,                # Overhead images can be upside-down
    degrees=15.0,              # Slight rotation for angled drone shots
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
    classes=[0],               # Person class only
    verbose=True,
)

## 3. Evaluate the Model

In [ ]:
# Validate on VisDrone val set
best_model = YOLO("runs/aerial/yolov8m-aerial/weights/best.pt")
metrics = best_model.val(data="VisDrone.yaml")

print(f"\nmAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

## 4. Download the Trained Model

Run this cell, then download `yolov8m-aerial.pt` from the Colab file browser.

**After downloading**, place it in your `resq_backend/` folder and restart your backend.

In [ ]:
import shutil

# Copy best weights to an easy-to-find location
src = "runs/aerial/yolov8m-aerial/weights/best.pt"
dst = "yolov8m-aerial.pt"
shutil.copy2(src, dst)
print(f"✅ Model saved as '{dst}'")
print(f"📥 Download it from the Colab file browser (left sidebar → Files)")
print(f"📂 Then place it in your resq_backend/ folder")

# Auto-download in Colab
try:
    from google.colab import files
    files.download(dst)
except ImportError:
    print("(Not in Colab — manually download the file)")